In [ ]:
!pip install -q rouge_score bert_score datasets

import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rouge_score import rouge_scorer
from bert_score import score as bert_score

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. Load summaries and references

In [ ]:
def clean(text):
    text = re.sub(r"</?S>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

refs_path = f"{RESULTS_DIR}/references.json"
if os.path.exists(refs_path):
    refs = json.load(open(refs_path))
else:
    # Rebuild the references: the first 200 papers of the arXiv test split 
    from datasets import load_dataset
    try:
        ds = load_dataset("ccdv/arxiv-summarization", split="test", streaming=True)
        test_raw = list(ds.take(200))
    except Exception:
        ds = load_dataset("ccdv/arxiv-summarization", split="test", streaming=True, revision="refs/convert/parquet")
        test_raw = list(ds.take(200))
    refs = [clean(e["abstract"]) for e in test_raw]
    json.dump(refs, open(refs_path, "w"))

files = {"BART-large-cnn (old)": "bart", "T5-base (old)": "t5", "Pegasus-cnn (old)": "pegasus",
         "Qwen2.5-0.5B zero-shot": "zeroshot", "Qwen2.5-0.5B fine-tuned": "finetuned"}
systems = {n: json.load(open(f"{RESULTS_DIR}/outputs_{k}.json")) for n, k in files.items()}
assert all(len(v) == len(refs) for v in systems.values()), "number of summaries and references differ"
print(len(refs), "papers,", len(systems), "systems")

## 2. Score

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rng = np.random.default_rng(0)

def ci(x, n=1000):
    x = np.asarray(x)
    return np.percentile([x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)], [2.5, 97.5])

rows, rougeL_per_example = [], {}
for name, outs in systems.items():
    r = [scorer.score(ref, out) for ref, out in zip(refs, outs)]
    rl = [s["rougeL"].fmeasure for s in r]
    _, _, f = bert_score(outs, refs, lang="en", batch_size=16, device="cuda")
    rougeL_per_example[name] = np.array(rl)
    lo, hi = ci(rl)
    rows.append({"System": name,
                 "ROUGE-1": np.mean([s["rouge1"].fmeasure for s in r]) * 100,
                 "ROUGE-2": np.mean([s["rouge2"].fmeasure for s in r]) * 100,
                 "ROUGE-L": np.mean(rl) * 100,
                 "ROUGE-L 95% CI": f"[{lo*100:.1f}, {hi*100:.1f}]",
                 "BERTScore F1": f.numpy().mean() * 100,
                 "Avg words": int(np.mean([len(o.split()) for o in outs]))})

results = pd.DataFrame(rows).round(1)
results.to_csv(f"{RESULTS_DIR}/results.csv", index=False)
print(results.to_string(index=False))

print("\nPaired ROUGE-L difference (fine-tuned minus other), 95% bootstrap CI:")
ft = rougeL_per_example["Qwen2.5-0.5B fine-tuned"]
for name in systems:
    if name != "Qwen2.5-0.5B fine-tuned":
        d = ft - rougeL_per_example[name]
        lo, hi = ci(d)
        print(f"  vs {name}: {d.mean()*100:+.1f}  [{lo*100:+.1f}, {hi*100:+.1f}]")

## 3. Chart and sample outputs

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results)); w = 0.27
ax.bar(x - w, results["ROUGE-1"], w, label="ROUGE-1")
ax.bar(x, results["ROUGE-2"], w, label="ROUGE-2")
ax.bar(x + w, results["ROUGE-L"], w, label="ROUGE-L")
ax.set_xticks(x); ax.set_xticklabels(results["System"], rotation=20, ha="right")
ax.set_ylabel("F1 (x100)"); ax.set_title("arXiv test set (200 papers)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/results_chart.png", dpi=150); plt.show()

# Fixed paper indices (not hand-picked)
with open(f"{RESULTS_DIR}/sample_outputs.md", "w") as f:
    for i in range(5):
        f.write(f"## Paper {i}\n\n**Reference:** {refs[i]}\n\n")
        for name, outs in systems.items():
            f.write(f"**{name}:** {outs[i]}\n\n")
        f.write("---\n\n")

# Bundle for download
!zip -q -r results.zip {RESULTS_DIR}
from google.colab import files as colab_files
colab_files.download("results.zip")